In [0]:
import requests,json
from datetime import datetime,timedelta,UTC
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [0]:
CONFIG = {
    "base_url": "https://hapi.fhir.org/baseR4",
    "resources": ["Patient", "Encounter", "Observation", "Condition"],
    "page_size": 100,        # records per page
    "pages_per_run": 3,      # number of pages to fetch per run (simulate 2-3 days)
    "base_path": "/Volumes/fhir_data/prd_raw/firstv/",
    "raw_path": "/Volumes/fhir_data/prd_raw/firstv/raw",
    "num_days" :3
}   

In [0]:
# generating Last N days

def get_run_dates(num_days):
    today = datetime.now(UTC)
    return [(today - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(num_days)]
print(get_run_dates(CONFIG['num_days']))
# creating folders

# Fetching Data from API using Pagination

In [0]:
def fetch_resource(config, resource, run_date):
    #parms = f"_count={config['page_size']}&_lastUpdated=ge{run_date}T00:00:00&_lastUpdated=lt{run_date}T23:59:59"
    start_date = f"{run_date}T00:00:00"
    end_date = (datetime.strptime(run_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%dT00:00:00")
    parms = f"_count={config['page_size']}&_lastUpdated=ge{start_date}&_lastUpdated=lt{end_date}"
    url = f"{config['base_url']}/{resource}?{parms}"
    all_entries = []
    page_num = 0
    api_urls_called = []
    print(f" [{run_date}] Fetching {resource} Data...")
    
    while page_num < config['pages_per_run']:
        try:
          response = requests.get(url, timeout=30)
          response.raise_for_status()
          data = response.json()
          entries = data.get('entry', [])
          all_entries.extend(entries)
          api_urls_called.append(url)
          print(f"Page {page_num +1}: fetched {len(entries)} records (total: {len(all_entries)})")
          # finding next page url for next iteration
          next_url = None
          for link in data.get('link',[]):
            if link.get('relation',[]) == 'next':
              next_url = link.get('url',[])
              break
          if not next_url:
            print(f" No more pages for {resource}")
            break
          url = next_url
          page_num += 1
        except requests.exceptions.RequestException as e:
          print(f"Error fetching data for {resource} page {page_num} : {e}")
          break
    # Metadata for the run
    result = {
      "resource_type": resource,
      "run_date": run_date,
      "extraction_timestamp": datetime.utcnow().isoformat(),
      "api_call_timestamp": datetime.utcnow().isoformat(),
      "api_urls_called": api_urls_called,
      "total_records": len(all_entries),
      "entries": all_entries
    }
    return result
          
    

# Saving Raw Data to DBFS

In [0]:
def raw_json(resource,data,config,run_date):
    raw_path = f"{config['raw_path']}/{resource.lower()}/date={run_date}"
    dbutils.fs.rm(raw_path, True)
    dbutils.fs.mkdirs(raw_path)
    file_path = f"{raw_path}/response_{run_date}.json"
    json_contnt = json.dumps(data,indent=2)
    dbutils.fs.put(file_path, json_contnt,overwrite=True)
    print(f"Raw data written to Json : {file_path}")
    return file_path   

# Main ingestion for All resources

In [0]:
#run_date = datetime.now().strftime("%Y-%m-%d")
run_dates = get_run_dates(CONFIG['num_days'])
print(f"Running incremental ingestion for Dates : {run_dates}")
print("=" * 50)

ingestion_log = []
for run_date in run_dates:
    print(f"Run date: {run_date}")
    print("=" * 50)
    for resource in CONFIG['resources']:
        print(f"Processing {resource}...")
        print("-" * 30)
        #fetching data from API

        data = fetch_resource(CONFIG, resource, run_date)

        # Saving raw json

        file_path = raw_json(resource,data,CONFIG,run_date)
        
        # log

        ingestion_log.append({
            "resource": resource,
            "file_path": file_path,
            "total_records": data['total_records'],
            "file_path": file_path,
            "status": "success"
        })
        print(f" Done : {data['total_records']} records written for {resource}")

print("=" * 50)
print("Ingestion Summary ")
print('=' * 50)


for i in ingestion_log:
    print(f"{i['resource']} : {i['total_records']} records | Status : {i['status']}")

# Verifying the Raw files 

In [0]:
print("Raw layer Contents:")
for resource in CONFIG['resources']:
  path = f"{CONFIG['raw_path']}/{resource.lower()}"
  try:
    files = dbutils.fs.ls(path)
    for f in files:
      print(f"File: {f.path}")
  except Exception as e :
    print(f" {resource} : {e}")